# Running Palace Simulations

[Palace](https://awslabs.github.io/palace/) is an open-source 3D electromagnetic simulator supporting eigenmode, driven (S-parameter), and electrostatic simulations. This notebook demonstrates using the `gsim.palace` API to run a driven simulation on custom RF components—specifically, the new stacked transformer generated using `gdsfactory`.

**Requirements:**
- IHP PDK: `uv pip install ihp-gdsfactory`
- `gsim` with Palace backend
- `circulax` (for lumped-element modeling and data fitting)

In [ ]:
import gdsfactory as gf
from gdsfactory.components.analog.transformers import stacked_transformer
import ihp

ihp.PDK.activate()

### Via unit cell adapter

In [ ]:
def ihp_via_unit(via_name: str, layer: str) -> gf.Component:
    """Adapter: converts one of IHP's VIA_RULES entries into the .info
    shape our _add_via_array()/_via_component_info() helpers expect."""
    rule = ihp.cells.VIA_RULES[via_name]
    size = rule["size"]
    pitch = size + rule["spacing"]
    c = gf.Component()
    c.add_polygon(
        [
            (-size / 2, -size / 2),
            (size / 2, -size / 2),
            (size / 2, size / 2),
            (-size / 2, size / 2),
        ],
        layer=layer,
    )
    c.info["xsize"] = size
    c.info["ysize"] = size
    c.info["enclosure"] = rule["enclosure"]
    c.info["column_pitch"] = pitch
    c.info["row_pitch"] = pitch
    return c

### Transformer layout

In [ ]:
stacked_transformer(
    layer_winding_primary="TopMetal2drawing",
    layer_crossing_primary="TopMetal1drawing",
    via_primary=ihp_via_unit("TopVia2", "TopVia2drawing"),
    layer_winding_secondary="Metal5drawing",
    layer_crossing_secondary="Metal4drawing",
    via_secondary=ihp_via_unit("Via4", "Via4drawing"),
).plot()

In [ ]:
cc = stacked_transformer(
    layer_winding_primary="TopMetal2drawing",
    layer_crossing_primary="TopMetal1drawing",
    via_primary=ihp_via_unit("TopVia2", "TopVia2drawing"),
    layer_winding_secondary="Metal5drawing",
    layer_crossing_secondary="Metal4drawing",
    via_secondary=ihp_via_unit("Via4", "Via4drawing"),
    add_pgs=True,
    layers_pgs=("Metal1drawing",),
)
cc.plot()

### Configure and run simulation with DrivenSim

In [ ]:
from gsim.palace import DrivenSim

# Create simulation object
sim = DrivenSim()

# Set output directory
sim.set_output_dir("./palace-sim-stacked_transformer")

# Set the component geometry
sim.set_geometry(cc)

# Configure layer stack from active PDK
sim.set_stack(substrate_thickness=180.0, include_substrate=True)

# Configure ports
sim.add_port(
    "P+", from_layer="metal1", to_layer="topmetal2", geometry="interlayer", excited=True
)
sim.add_port(
    "P-", from_layer="metal1", to_layer="topmetal2", geometry="interlayer", excited=True
)
sim.add_port(
    "S+", from_layer="metal1", to_layer="metal5", geometry="interlayer", excited=True
)
sim.add_port(
    "S-", from_layer="metal1", to_layer="metal5", geometry="interlayer", excited=True
)

# Configure driven simulation (frequency sweep for S-parameters)
sim.set_driven(fmin=10e9, fmax=150e9, num_points=50)

# Validate configuration
print(sim.validate_config())

In [ ]:
# Generate mesh (presets: "coarse", "default", "fine")
sim.set_airbox(margin_x=50, margin_y=50, z_above=50, z_below=5)
sim.mesh(preset="default", refined_mesh_size=1.5)
sim.write_config()

In [ ]:
sim.plot_mesh(show_groups=["metal", "via", "P"])

In [ ]:
sim.plot_mesh(
    style="solid",
    transparent_groups=["air__None", "sio2__None", "air__sio2"],
)

### Run simulation on cloud

In [ ]:
# Run simulation on GDSFactory+ cloud
results = sim.run()

In [ ]:
results.plot_interactive()

In [ ]:
results.plot_interactive(phase=True)

In [ ]:
results.plot()